In [0]:
# API keys
import os
from dotenv import load_dotenv

load_dotenv()

COINGECKO_API_KEY = os.getenv("COINGECKO_API_KEY")
COINCAP_API_KEY = os.getenv("COINCAP_API_KEY")

In [0]:
import requests
import pandas as pd
from datetime import datetime

def fetch_coingecko(api_key, limit=50):
    url = "https://api.coingecko.com/api/v3/coins/markets"
    headers = {"accept": "application/json", "x-cg-demo-api-key": api_key}
    params = {
        "vs_currency": "usd",
        "order": "market_cap_desc",
        "per_page": limit,
        "page": 1,
        "sparkline": "false"
    }
    response = requests.get(url, headers=headers, params=params)
    response.raise_for_status()
    data = response.json()
    
    df = pd.DataFrame(data)
    cols = ["id", "symbol", "name", "current_price", "market_cap", 
            "total_volume", "price_change_percentage_24h", "high_24h", 
            "low_24h", "last_updated"]
    df = df[cols]
    df = df.rename(columns={
        "current_price": "price_usd",
        "total_volume": "volume_24h",
        "price_change_percentage_24h": "percent_change_24h"
    })
    return df

In [0]:
def fetch_coincap(api_key, limit=5):
    url = "https://rest.coincap.io/v3/assets"
    headers = {"Authorization": f"Bearer {api_key}"}
    params = {"limit": limit}
    response = requests.get(url, headers=headers, params=params)
    response.raise_for_status()
    data = response.json()["data"]
    
    df = pd.DataFrame(data)
    cols = ["id", "symbol", "name", "priceUsd", "marketCapUsd", 
            "changePercent24Hr", "volumeUsd24Hr", "supply", "rank"]
    df = df[cols]
    df = df.rename(columns={
        "priceUsd": "price_usd",
        "marketCapUsd": "market_cap_usd",
        "changePercent24Hr": "percent_change_24h",
        "volumeUsd24Hr": "volume_24h"
    })
    numeric_cols = ["price_usd", "market_cap_usd", "percent_change_24h", "volume_24h", "supply", "rank"]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df

In [0]:
from pyspark.sql.functions import current_timestamp, lit

df_cg_pd2 = fetch_coingecko(COINGECKO_API_KEY, limit=50)
df_cg2 = spark.createDataFrame(df_cg_pd2)
df_cg2 = df_cg2.withColumn("ingestion_ts", current_timestamp()) \
               .withColumn("source_file", lit("coingecko_batch_2"))
df_cg2.write.format("delta").mode("append").save("/Volumes/cryptolake/bronze/data/coingecko")
print("CoinGecko batch 2 añadido")

df_cc_pd2 = fetch_coincap(COINCAP_API_KEY, limit=5)
df_cc2 = spark.createDataFrame(df_cc_pd2)
df_cc2 = df_cc2.withColumn("ingestion_ts", current_timestamp()) \
               .withColumn("source_file", lit("coincap_batch_2"))
df_cc2.write.format("delta").mode("append").save("/Volumes/cryptolake/bronze/data/coincap")
print("CoinCap batch 2 añadido")